# Tasdeeq - GPU evaluation (Engine A vs B vs pipeline)

**Before anything:** `Runtime -> Change runtime type -> T4 GPU` (free tier is fine).

This notebook runs the spec-§8 setups on the 20 held-out invoices:
1. **Engine A alone** - Qaari (Qwen2-VL-2B + LoRA adapter, now verified to load)
2. **Engine B alone** - EasyOCR (ur+en)
3. **Fallback** - Tesseract (urd+eng)
4. **Full verified pipeline** - A+B + verification layer

Steps: upload `tasdeeq_colab.zip` from your repo root when prompted -> Run all.
First run downloads ~5 GB of models (~3-6 min), then ~5-10 min for inference.
Results end up in `results/results.md`, `results/results.json`, `results/raw_cache.json` (auto-downloaded at the end).


In [ ]:
from google.colab import files
uploaded = files.upload()  # choose tasdeeq_colab.zip
!unzip -q -o tasdeeq_colab.zip -d /content/tasdeeq
%cd /content/tasdeeq
!ls

In [ ]:
!pip install -q -r requirements.txt
!pip uninstall -y torchao > /dev/null  # Colab's torchao 0.10 breaks peft>=0.21 (optional backend)
!apt-get -qq install -y tesseract-ocr tesseract-ocr-urd > /dev/null  # fallback engine
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch runtime!')


In [ ]:
# Three setups in one pass: each engine read once, then each alone + pipeline (A+B).
# Raw OCR lines are cached to results/raw_cache.json (downloaded next cell) for free local re-scoring.
!python -m eval.run_eval --engines qaari,easyocr,tesseract --out results/results.md


In [ ]:
from IPython.display import Markdown
display(Markdown(open('results/results.md', encoding='utf-8').read()))
files.download('results/results.md')
files.download('results/results.json')
files.download('results/raw_cache.json')
